# V13 — Symbolic Intermediate Representation SFT (Nemotron Challenge)

## Goal
Build a symbolic IR training layer: parse → transform → constraint solve → boxed answer.

In [ ]:
import pandas as pd
import random
import re
from pathlib import Path

TRAIN_PATH = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pd.read_csv(TRAIN_PATH)
print(df.shape)

In [ ]:
# -------------------------------
# Step 1: Category detection
# -------------------------------

def categorize(p):
    p = str(p).lower()
    if 'bit' in p: return 'bit'
    if 'cipher' in p: return 'cipher'
    if 'unit' in p: return 'unit'
    if 'gravity' in p: return 'gravity'
    if 'equation' in p: return 'equation'
    return 'other'

df['cat'] = df['prompt'].apply(categorize)
df['cat'].value_counts()

In [ ]:
# -------------------------------
# Step 2: Symbolic IR generator
# -------------------------------

def make_symbolic_trace(prompt, answer):
    # Minimal IR template (NOT full CoT)
    return f'''[PARSE]
{prompt[:120]}

[SYMBOL_MAP]
A->α B->β OP->φ

[CONSTRAINTS]
match_rules=True consistency=True

[EXEC]
apply_transform

[OUTPUT]
\boxed{{{answer}}}'''

In [ ]:
# -------------------------------
# Step 3: Build synthetic IR dataset
# -------------------------------

N_SYNTH = 300
cats = ['bit','cipher','unit','gravity','equation']

synthetic = []
for c in cats:
    sub = df[df['cat']==c].sample(min(N_SYNTH, len(df[df['cat']==c])), random_state=42)
    for _, r in sub.iterrows():
        synthetic.append({
            'prompt': r['prompt'],
            'answer': r['answer'],
            'text': make_symbolic_trace(r['prompt'], r['answer'])
        })

print('synthetic size:', len(synthetic))

In [ ]:
# -------------------------------
# Step 4: Merge training set
# -------------------------------

train_rows = df.to_dict('records') + synthetic
print('total rows:', len(train_rows))

In [ ]:
# -------------------------------
# Step 5: Training config (V2 warmstart)
# -------------------------------

CONFIG = {
    'warmstart': '/kaggle/input/notebooks/jatalepawan/v2-end-to-end-finetuning-lb082-bit-reweight/submission.zip',
    'lr': 7e-7,
    'steps': 56,
    'reset_weights': False
}

print(CONFIG)

## Notes
This V13 does NOT implement full solver correctness.
It encodes symbolic structure as an intermediate representation for SFT.